# 03. Hybrid 分支训练（FP32）

第二个分支：**Hybrid**（`HybridSNN`），看的是原始 16 通道波形而不是聚合统计特征——
和 Context 分支是完全不同的"视角"，这正是教程 04 章"多分支互补"的真实例子。
用的还是同一套真实项目代码（`semg_snn_90_loop/`）和真实数据。

In [ ]:
import sys, random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path("training/semg_snn_90_loop")
sys.path.insert(0, str(PROJECT_ROOT))

from train import EMGDataset
from model import HybridSNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NB_RUNS = PROJECT_ROOT / "runs_notebook"
NB_RUNS.mkdir(exist_ok=True)

## 1. 结构：ConvLIF + Spiking Jaccard Attention

结构（`model.py` 的 `HybridSNN` / `ConvLIFBranch`）：

```
feature_current = Linear(336, 384) -> LayerNorm(384)        # 统计特征这一路
conv 分支（看原始波形）:
  Conv1d(16->64, k=7) -> BatchNorm -> GELU -> Conv1d(64->128, k=5) -> BatchNorm
  -> 逐时间步 LIF 积分发放，得到 128 维二值脉冲序列
  -> 1x1 卷积各自算出 query/key/value（也是二值脉冲）
  -> Jaccard 相似度注意力: intersection/union（交集计数 / 并集计数,这里还是浮点除法,
     04(量化)会把它换成查找表)
  -> value * attention + 原始脉冲，按时间平均池化
融合: concat(sf, conv_pooled) -> Linear(512,256) -> LayerNorm -> LIF -> out
```

这个分支比 Context 分支多两种算子（GELU、BatchNorm），量化的时候（下一个 notebook）
要多做两步替换。

In [ ]:
model = HybridSNN(features=336).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Hybrid 模型参数量: {n_params:,}")
del model

## 2. 训练循环

和 Context 分支同样的训练工程（AdamW + 余弦学习率 + 类别加权 CE + label smoothing +
梯度裁剪 + 早停），**没有热启动**——真实项目里 `hybrid_sja_v1` 就是直接从随机初始化
训练出来的。超参数取自项目 `metrics.json` 里记录的真实值：
`epochs=30, lr=1e-3, batch_size=128, patience=7, weight_power=0.15, label_smoothing=0.04`。

In [ ]:
@torch.no_grad()
def evaluate_hybrid(model, loader):
    model.eval()
    preds, targets = [], []
    for f, raw, y, subject in loader:
        output, _ = model(f.to(device), raw.to(device), subject.to(device))
        preds.extend(output.argmax(1).cpu().tolist())
        targets.extend(y.tolist())
    y_arr, p_arr = np.asarray(targets), np.asarray(preds)
    return {
        "accuracy": accuracy_score(y_arr, p_arr),
        "macro_f1": f1_score(y_arr, p_arr, average="macro"),
        "gesture_accuracy": float(np.mean(p_arr[y_arr != 0] == y_arr[y_arr != 0])),
    }


def train_hybrid_model(run_name: str, epochs: int, lr: float, patience: int,
                        batch_size: int = 128, weight_power: float = 0.15,
                        label_smoothing: float = 0.04, seed: int = 42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    sets = {
        split: EMGDataset(
            PROJECT_ROOT / "data" / f"{split}.npz", PROJECT_ROOT / "data" / "normalization.npz",
            split == "train", context=1,
        )
        for split in ("train", "val", "test")
    }
    loaders = {
        "train": DataLoader(sets["train"], batch_size, shuffle=True, num_workers=4, pin_memory=True),
        "val": DataLoader(sets["val"], batch_size * 2, num_workers=4, pin_memory=True),
        "test": DataLoader(sets["test"], batch_size * 2, num_workers=4, pin_memory=True),
    }
    model = HybridSNN(sets["train"].features.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    counts = np.bincount(sets["train"].y, minlength=13)
    class_weights = torch.tensor((counts.sum() / (13 * counts)) ** weight_power,
                                  dtype=torch.float32, device=device)

    run_dir = NB_RUNS / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    best_acc, stale = -1.0, 0

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for f, raw, y, subject in loaders["train"]:
            f, raw, y, subject = f.to(device), raw.to(device), y.to(device), subject.to(device)
            optimizer.zero_grad(set_to_none=True)
            output, _ = model(f, raw, subject)
            loss = nn.functional.cross_entropy(output, y, weight=class_weights,
                                                label_smoothing=label_smoothing)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()
            losses.append(loss.item())
        scheduler.step()

        val_metrics = evaluate_hybrid(model, loaders["val"])
        print(f"epoch {epoch:02d}  loss={np.mean(losses):.4f}  val_acc={val_metrics['accuracy']:.4f}")
        if val_metrics["accuracy"] > best_acc:
            best_acc, stale = val_metrics["accuracy"], 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "validation": val_metrics},
                       run_dir / "best.pt")
        else:
            stale += 1
            if stale >= patience:
                print("early stopping"); break

    ckpt = torch.load(run_dir / "best.pt", map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    test_metrics = evaluate_hybrid(model, loaders["test"])
    print(f"\n=== {run_name} 最终结果（第 {ckpt['epoch']} 轮）===")
    print(f"val:  accuracy={ckpt['validation']['accuracy']:.4f}")
    print(f"test: accuracy={test_metrics['accuracy']:.4f}  macro_f1={test_metrics['macro_f1']:.4f}  "
          f"gesture_accuracy={test_metrics['gesture_accuracy']:.4f}")
    return run_dir / "best.pt", test_metrics

In [ ]:
# 首次运行建议先把 epochs 调小做 smoke test；这里用的是项目实际记录的超参数
hybrid_checkpoint, hybrid_test_metrics = train_hybrid_model(
    run_name="hybrid_sja_nb", epochs=30, lr=1e-3, patience=7,
    batch_size=128, weight_power=0.15, label_smoothing=0.04,
)
print("\n参考值(真实项目 hybrid_sja_v1, RESULTS.md 单窗协议): "
      "accuracy=0.8876 macro_f1=0.7615 gesture_accuracy=0.7242")

## 下一步

打开 [04_hybrid_quantization.ipynb](04_hybrid_quantization.ipynb) 把这个 checkpoint
改造成硬件友好版本——这个分支要处理的算子替换比 Context 多（BatchNorm 折叠、
GELU→ReLU6、Jaccard 除法→查找表）。